In [ ]:
################# Adamts4 project, to be submitted to NatComm ####################
######################### 0_Integration and cleanup   ##########################

In [ ]:
import os
import math
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scanpy as sc
import scanpy.external as sce
import harmonypy as hm
import bbknn
import scvi
import gseapy as gp
from scipy import sparse
from scipy.stats import zscore
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.sparse import issparse
from sklearn.preprocessing import LabelEncoder
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.pyplot import rc_context

warnings.filterwarnings("ignore")

sc.settings.verbosity = 3
sc.settings.n_jobs = os.cpu_count()
sc.set_figure_params(figsize=(10, 10), frameon=False)
scv.set_figure_params()
plt.rcParams['axes.grid'] = False

plot_directory = '/Users/aly/Downloads/2024_Ali_scRNAseq/2024_Mahsa_scRNAseq/Analysis/0_260318_Final'
os.makedirs(plot_dir, exist_ok=True)

In [ ]:
# Define the directory where plots will be saved
plot_directory = '/Users/aly/Downloads/2024_Mahsa_scRNAseq/Analysis/20241129_Bleo_reintegrated/Integrated/'

# Ensure the directory exists (optional)
import os
os.makedirs(plot_directory, exist_ok=True)

In [ ]:
raw_path = "/Users/aly/Downloads/2024_Mahsa_scRNAseq/RawData"
h5ad_files = [f for f in os.listdir(raw_path) if f.endswith('.h5ad')]

print("List of .h5ad files:")
for f in h5ad_files: print(f)

In [ ]:
desired_order = ["1.Saline.h5ad", "2.Bleo_d14.h5ad", "3.Bleo_d30.h5ad", "4.Bleo_d60.h5ad"]
adatas = []

for f in desired_order:
    path = os.path.join(raw_path, f)
    if os.path.exists(path):
        ad = sc.read_h5ad(path)
        ad.var_names_make_unique()
        ad.obs['sample'] = f.split('.', 1)[1].replace('.h5ad', '')
        adatas.append(ad)

if adatas:
    adata = adatas[0].concatenate(adatas[1:], join='outer')
    sample_order = [f.split('.', 1)[1].replace('.h5ad', '') for f in desired_order]
    adata.obs['sample'] = pd.Categorical(adata.obs['sample'], categories=sample_order, ordered=True)
    adata = adata[adata.obs['sample'].argsort()].copy()
    
    print(adata)
    print(adata.obs['sample'].value_counts())

In [ ]:
print(adata.obs['sample'].value_counts())

In [ ]:
tmp = adata
tmp

In [ ]:
tmp = adata
tmp

In [ ]:
# mitochondrial genes
tmp.var['mt'] = tmp.var_names.str.startswith('mt-') 
# ribosomal genes
tmp.var['ribo'] = tmp.var_names.str.startswith(("Rps","Rpl"))
# hemoglobin genes.
tmp.var['hb'] = tmp.var_names.str.contains(("^Hb[^(P)]"))
sc.pp.calculate_qc_metrics(tmp, qc_vars=['mt','ribo','hb'], percent_top=None, log1p=False, inplace=True)
tmp.var['Ptprc'] = tmp.var_names.str.contains('Ptprc')
myGene ='wpre' #tdTomato 
tmp.var[myGene] = tmp.var_names.str.contains(myGene)

In [ ]:
sc.pl.violin(tmp, [myGene],jitter=0.4, groupby = 'sample', rotation= 90)

In [ ]:
sc.pl.violin(tmp, ['n_genes_by_counts'],jitter=0.4, groupby = 'sample', rotation= 90)

In [ ]:
sc.pl.violin(tmp, ['total_counts'],jitter=0.4, groupby = 'sample', rotation= 90)

In [ ]:
sc.pl.violin(tmp, ['pct_counts_mt'],jitter=0.4, groupby = 'sample', rotation= 90)

In [ ]:
sc.pl.violin(tmp, ['Ptprc'],jitter=0.4, groupby = 'sample', rotation= 90)

In [ ]:
sc.pl.scatter(tmp, x='total_counts', y='pct_counts_mt', color="sample")

In [ ]:
df = pd.DataFrame(tmp.obs['sample'].value_counts()).reset_index()
df.columns = ['Sample', 'Count']  
for index, row in df.iterrows():
    k = row['Sample']
    p1 = sc.pl.scatter(tmp, x='total_counts', y='n_genes_by_counts', color='pct_counts_mt', groups=[k])

In [ ]:
import seaborn as sb
import matplotlib.pyplot as plt

for index, row in df.iterrows():
    k = row['Sample']
    print(k)

    p3 = sb.histplot(tmp[tmp.obs['sample'] == k].obs['total_counts'], kde=False, bins=30)
    plt.title(f'Distribution of Total Counts for {k}')
    plt.xlabel('Total Counts')
    plt.ylabel('Frequency')
    plt.show()

    p6 = sb.histplot(tmp[tmp.obs['sample'] == k].obs['n_genes_by_counts'], kde=False, bins=60)
    plt.title(f'Distribution of Genes by Counts for {k}')
    plt.xlabel('Number of Genes by Counts')
    plt.ylabel('Frequency')
    plt.show()

In [ ]:
for index, row in df.iterrows():
    k = row['Sample']
    print(k)
    sc.pl.highest_expr_genes(tmp[tmp.obs['sample']==k], n_top=20, )

In [ ]:
# Filter cells according to identified QC thresholds:
print('Total number of cells: {:d}'.format(tmp.n_obs))

sc.pp.filter_cells(tmp, min_counts = 1200)
print('Number of cells after min count filter: {:d}'.format(tmp.n_obs))

sc.pp.filter_cells(tmp, max_counts = 40000)
print('Number of cells after max count filter: {:d}'.format(tmp.n_obs))

adata = tmp[tmp.obs['pct_counts_mt'] < 20]
adata = tmp[tmp.obs['pct_counts_mt'] < 20] #### change to tmp!!!
print('Number of cells after MT filter: {:d}'.format(tmp.n_obs))

sc.pp.filter_cells(tmp, min_genes = 600)
print('Number of cells after gene filter: {:d}'.format(tmp.n_obs))
print('Total number of genes: {:d}'.format(tmp.n_vars))

sc.pp.filter_genes(tmp, min_cells=20)
print('Number of genes after cell filter: {:d}'.format(tmp.n_vars))

In [ ]:
print(tmp.obs['sample'].value_counts())

In [ ]:
import scrublet as scr

scrub = scr.Scrublet(tmp.X)
scores, preds = scrub.scrub_doublets()
threshold = 0.25  
scrub.threshold_ = threshold
preds = scores > threshold

tmp.obs['doublet_scores'] = scores
tmp.obs['predicted_doublets'] = preds
scrub.plot_histogram()

In [ ]:
print(df.columns)

In [ ]:
mycluster = "leiden_0.8"

for index, row in df.iterrows():
    sample_name = row['Sample']
    print(sample_name)
    
    sub = tmp[tmp.obs['sample'] == sample_name].copy()
    
    sc.pp.filter_genes(sub, min_cells=1)
    sc.pp.normalize_per_cell(sub, counts_per_cell_after=1e4)
    sc.pp.log1p(sub)
    
    sc.pp.highly_variable_genes(sub, min_mean=0.0125, max_mean=3, min_disp=0.5)
    sc.pp.pca(sub, n_comps=50, use_highly_variable=True, svd_solver='arpack')
    sc.pp.neighbors(sub)
    sc.tl.umap(sub)
    sc.tl.leiden(sub, key_added=mycluster)
    
    sc.pl.umap(sub, color=[mycluster])

In [ ]:
sc.pp.filter_genes(tmp, min_cells=20)
tmp.raw = tmp
sc.pp.normalize_per_cell(tmp, counts_per_cell_after=1e4)
tmp

In [ ]:
# logaritmize
sc.pp.log1p(tmp)

In [ ]:
sc.pp.highly_variable_genes(tmp, flavor='cell_ranger', n_top_genes=500)
print('\n','Number of highly variable genes: {:d}'.format(np.sum(tmp.var['highly_variable'])))

In [ ]:
sc.pl.highly_variable_genes(tmp)

In [ ]:
sc.pp.pca(tmp, n_comps=20, use_highly_variable=True, svd_solver='arpack')
sce.pp.harmony_integrate(tmp, 'sample',max_iter_harmony=30)
sc.pp.neighbors(tmp,use_rep = 'X_pca_harmony')#,n_pcs =0,n_neighbors=15)
sc.tl.umap(tmp)

In [ ]:
sc.pl.umap(tmp, color='n_counts')

In [ ]:
sc.tl.leiden(tmp, key_added = "leiden_0.8") # default resolution in 1.0
sc.tl.leiden(tmp, resolution = 0.6, key_added = "leiden_0.6")
sc.tl.leiden(tmp, resolution = 1.0, key_added = "leiden_1.0")
sc.tl.leiden(tmp, resolution = 0.4, key_added = "leiden_0.4")
sc.tl.leiden(tmp, resolution = 1.4, key_added = "leiden_1.4")

In [ ]:
sc.pl.violin(tmp,"Ptprc",groupby="leiden_0.6")

In [ ]:
sc.pl.violin(tmp,"Epcam",groupby="leiden_0.6")

In [ ]:
sc.pl.violin(tmp,"Pecam1",groupby="leiden_0.6")

In [ ]:
sc.pl.violin(tmp,"Sfrp1",groupby="leiden_0.6")

In [ ]:
unique_samples = tmp.obs['sample'].unique()
print("Unique samples in adata:", unique_samples)

In [ ]:
### remove Ptprc and Pecam1+ and Epithelial cells cells (contamination) 

tmp = tmp[-tmp.obs["leiden_0.6"].isin(['8','3','11'])]

In [ ]:
if 'X_scaled' in tmp.layers:
    del tmp.layers['X_scaled']
    
if 'scaled' in tmp.layers:
    del tmp.layers['scaled']
# Save the updated tmp object
output_path = '/Users/aly/Downloads/2024_Ali_scRNAseq/2024_Mahsa_scRNAseq/ScanpyObjects/241129_Bleo_Harmony_v2.h5ad'
tmp.write(output_path)